In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

def main():
    # ---- 1. 打开数据（不立即 load） ----
    ds = xr.open_dataset(MERGED_FILE, chunks={})
    
    # ---- 2. 【先抽 year】减少时间维度 ----
    ds_0008 = ds.sel(time=ds.time.dt.year == 8)

    # ---- 3. 【再抽 lat】减少空间维度 ----
    # 只要 60–90N，不要整球
    T_cap = ds_0008["T"].sel(lat=slice(60, 90))  # (time, plev, lat, lon)

    # ---- 4. 【再纬向平均】 ----
    T_zm = T_cap.mean("lon")  # (time, plev, lat)

    # ---- 5. 【极区最冷格点】 ----
    T_min = T_zm.min("lat")  # (time, plev)

    # ---- 6. 压层选择 ----
    lev = T_min["plev"].values
    lev_target = 5000.0  # Pa
    lev_idx = int(np.abs(lev - lev_target).argmin())

    T_series = T_min.isel(plev=lev_idx)

    # ---- 7. 画图 ----
    doy = T_series.time.dt.dayofyear.values

    plt.figure()
    plt.plot(doy, T_series.values, "-o", ms=3)
    plt.xlabel("Day of year (0008)")
    plt.ylabel(f"T_min at ~{lev[lev_idx]/100:.1f} hPa (K)")
    plt.title("Quick check (FAST): Year 0008 min T(60–90N)")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    ds.close()


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
============================================================
BlockA (NEW) — Compute v'T' FULL 4D + 45–75N mean
------------------------------------------------------------
Outputs:
  1. EHF_EXTR_vTprime_4D.nc
  2. EHF_MERGED_vTprime_4D.nc
  3. EHF_EXTR_45_75N_lev_time.nc
  4. EHF_MERGED_45_75N_lev_time.nc
============================================================
"""

import os
import glob
import numpy as np
import xarray as xr

# ------------------------- Paths -------------------------
ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(ROOT, exist_ok=True)

EXTR_V_DIR = "/home/weiji/restart_exam/longrun_withchem_data/V"
EXTR_T_DIR = "/home/weiji/restart_exam/longrun_withchem_data/T"

MERGED_INT_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

# Output files
EXTR_FULL_NC   = os.path.join(ROOT, "EHF_EXTR_vTprime_4D.nc")
MERGED_FULL_NC = os.path.join(ROOT, "EHF_MERGED_vTprime_4D.nc")

EXTR_45_75_NC   = os.path.join(ROOT, "EHF_EXTR_45_75N_lev_time.nc")
MERGED_45_75_NC = os.path.join(ROOT, "EHF_MERGED_45_75N_lev_time.nc")


# ==========================================================
# Helper: area mean 45–75N
# ==========================================================
def area_mean_45_75N(da):
    if "lat" not in da.dims:
        raise ValueError("lat not in dims")
    da_sel = da.sel(lat=slice(45, 75))
    w = np.cos(np.deg2rad(da_sel["lat"]))
    return da_sel.weighted(w).mean("lat")


# ==========================================================
# Compute v'T' for EXTR (daily files)
# ==========================================================
def compute_EXTR_full_vTprime():
    v_files = sorted(glob.glob(os.path.join(EXTR_V_DIR, "*.V.isobar.nc")))
    t_files = sorted(glob.glob(os.path.join(EXTR_T_DIR, "*.T.isobar.nc")))
    if len(v_files) == 0:
        raise FileNotFoundError("No EXTR V files")
    if len(t_files) == 0:
        raise FileNotFoundError("No EXTR T files")

    dsV = xr.open_mfdataset(v_files, combine="by_coords", parallel=False, decode_cf=True)
    dsT = xr.open_mfdataset(t_files, combine="by_coords", parallel=False, decode_cf=True)

    V = dsV["V"]  # time lev lat lon
    T = dsT["T"]

    # Ensure same time
    V, T = xr.align(V, T, join="inner")

    # Zonal mean
    V_zm = V.mean("lon")
    T_zm = T.mean("lon")

    # Eddy components
    Vp = V - V_zm
    Tp = T - T_zm

    # FULL v'T'
    vt_full = Vp * Tp  # (time, lev, lat, lon)

    # Region mean for BlockB/C/D
    vt_zm = vt_full.mean("lon")  # first remove lon for area averaging
    vt_45_75 = area_mean_45_75N(vt_zm)  # (time, lev)

    # lev unit fix
    vt_45_75 = vt_45_75.transpose("time", "lev")
    vt_45_75["lev"].attrs["units"] = "hPa"

    # Save files
    vt_full.to_netcdf(EXTR_FULL_NC)
    vt_45_75.to_netcdf(EXTR_45_75_NC)

    dsV.close(); dsT.close()
    print("[BlockA] Saved EXTR FULL + 45–75N mean")


# ==========================================================
# Compute v'T' for MERGED (INT file)
# ==========================================================
def compute_MERGED_full_vTprime():
    ds = xr.open_dataset(MERGED_INT_PATH)

    V = ds["V"]  # time plev lat lon
    T = ds["T"]

    # Clean missing values
    V = V.where(np.abs(V) < 1e20)
    T = T.where(np.abs(T) < 1e20)

    V_zm = V.mean("lon")
    T_zm = T.mean("lon")
    Vp = V - V_zm
    Tp = T - T_zm

    vt_full = Vp * Tp  # FULL 4D

    # Region mean (need to remove lon first)
    vt_45_75 = area_mean_45_75N(vt_full.mean("lon"))
    vt_45_75 = vt_45_75.rename({"plev": "lev"}).transpose("time", "lev")
    vt_45_75["lev"].attrs["units"] = "Pa"

    # Save
    vt_full.to_netcdf(MERGED_FULL_NC)
    vt_45_75.to_netcdf(MERGED_45_75_NC)

    ds.close()
    print("[BlockA] Saved MERGED FULL + 45–75N mean")


# ==========================================================
# Main
# ==========================================================
def main():
    compute_EXTR_full_vTprime()
    compute_MERGED_full_vTprime()
    print("[BlockA] Done.")


if __name__ == "__main__":
    main()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
============================================================
BlockB (NEW) — Timeseries of Eddy Heat Flux (45–75°N)
------------------------------------------------------------
Compatible with new BlockA outputs:

  - EHF_EXTR_45_75N_lev_time.nc   (baseline)
  - EHF_MERGED_45_75N_lev_time.nc (special years)

This block remains identical in function:
  ✓ low-10 vs others       (raw + RM)
  ✓ special years vs climatology (raw + RM)
  ✓ Levels: 10 / 50 / 100 hPa
============================================================
"""

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# ---------------- paths ----------------
O3_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

# Input files from new BlockA
EHF_EXTR_NC   = os.path.join(EHF_ROOT, "EHF_EXTR_45_75N_lev_time.nc")
EHF_MERGED_NC = os.path.join(EHF_ROOT, "EHF_MERGED_45_75N_lev_time.nc")

# O3 ranking
LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)
RM_WIN = 15


# ==========================================================
# Running mean 1D
# ==========================================================
def running_mean_1d(arr, win=15):
    n = arr.size
    k = win // 2
    out = np.full(n, np.nan)
    for i in range(n):
        i0 = max(0, i-k)
        i1 = min(n, i+k+1)
        out[i] = np.nanmean(arr[i0:i1])
    return out


def get_level_index_pa(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx


# ==========================================================
# Build composite
# ==========================================================
def build_composite(var, years, n_days):
    comp_list = []
    used = []
    yrs_all = var.time.dt.year.values.astype(int)

    for y in years:
        if (y not in yrs_all) or ((y - 1) not in yrs_all):
            continue

        cur = var.sel(time=var.time.dt.year == y)
        prev = var.sel(time=var.time.dt.year == y-1)

        if cur.time.size < n_days or prev.time.size < n_days:
            continue

        seg = np.concatenate([
            np.array(prev)[n_days - N_PREV:n_days],
            np.array(cur)[0:n_days - N_PREV]
        ])
        comp_list.append(seg)
        used.append(y)

    if len(comp_list) == 0:
        return None, []

    return np.stack(comp_list, axis=0), np.array(used)


# ==========================================================
# Plot 1: low10 vs others (raw)
# ==========================================================
def plot_low10_vs_others_raw(var_extr, low10, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values)
    n_days = int(var_extr.time.dt.dayofyear.max())

    low = np.array(low10)
    others = years_extr[~np.isin(years_extr, low)]

    comp_low, used_low = build_composite(var_extr, low, n_days)
    comp_oth, used_oth = build_composite(var_extr, others, n_days)

    if comp_low is None or comp_oth is None:
        print("[B] Not enough data for low10 raw plot.")
        return

    oth_mean = np.nanmean(comp_oth, axis=0)
    oth_std  = np.nanstd(comp_oth, axis=0)

    fig, ax = plt.subplots(1, 1, figsize=(18, 4), constrained_layout=True)
    x = np.arange(n_days)
    n_ext = comp_low.shape[0]
    cols = cm.twilight(np.linspace(0, 1, n_ext))

    for i in range(n_ext):
        ax.plot(x, comp_low[i], color=cols[i], lw=1.6)

    ax.fill_between(x, oth_mean - oth_std, oth_mean + oth_std,
                    color="0.7", alpha=0.35)
    ax.plot(x, oth_mean, "k--", lw=2)

    ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
    ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar",
                        "Apr","May","Jun","Jul","Aug","Sep"])
    ax.set_ylabel(f"EHF 45–75°N ({level_label})")
    ax.set_title("Low10 vs Others (raw)")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close()


# ==========================================================
# Plot 1: low10 vs others (RM)
# ==========================================================
def plot_low10_vs_others_RM(var_extr, low10, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values)
    n_days = int(var_extr.time.dt.dayofyear.max())

    low = np.array(low10)
    others = years_extr[~np.isin(years_extr, low)]

    comp_low, used_low = build_composite(var_extr, low, n_days)
    comp_oth, used_oth = build_composite(var_extr, others, n_days)

    if comp_low is None or comp_oth is None:
        print("[B] Not enough data for low10 RM plot.")
        return

    oth_mean = np.nanmean(comp_oth, axis=0)
    oth_std  = np.nanstd(comp_oth, axis=0)

    oth_mean_rm = running_mean_1d(oth_mean, RM_WIN)
    oth_std_rm  = running_mean_1d(oth_std,  RM_WIN)

    fig, ax = plt.subplots(1, 1, figsize=(18, 4), constrained_layout=True)
    x = np.arange(n_days)
    n_ext = comp_low.shape[0]
    cols = cm.twilight(np.linspace(0,1,n_ext))

    for i in range(n_ext):
        rm = running_mean_1d(comp_low[i], RM_WIN)
        ax.plot(x, rm, color=cols[i], lw=2.0)

    ax.fill_between(x, oth_mean_rm - oth_std_rm, oth_mean_rm + oth_std_rm,
                    color="0.7", alpha=0.5)
    ax.plot(x, oth_mean_rm, "k-", lw=2)

    ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
    ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar",
                        "Apr","May","Jun","Jul","Aug","Sep"])
    ax.set_ylabel(f"EHF 45–75°N ({level_label})")
    ax.set_title("Low10 vs Others (15d RM)")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close()


# ==========================================================
# Plot 2: special vs climatology (RM only)
# ==========================================================
def plot_special_vs_clim_RM(var_extr, var_merged, low25, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values)
    n_days = int(var_extr.time.dt.dayofyear.max())

    # Baseline (raw)
    comp_all, used_all = build_composite(var_extr, years_extr, n_days)
    if comp_all is None:
        print("[B] No EXTR climatology available.")
        return

    all_mean = np.nanmean(comp_all, axis=0)
    all_std  = np.nanstd(comp_all, axis=0)

    # Special years (RM)
    yrs_mer = np.unique(var_merged.time.dt.year.values)

    fig, ax = plt.subplots(1,1, figsize=(18,4), constrained_layout=True)
    x = np.arange(n_days)
    cols = plt.cm.tab10(np.linspace(0,1,len(YEARS_SPECIAL)))

    for i, y in enumerate(YEARS_SPECIAL):
        if y not in yrs_mer:
            continue
        cur = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == y-1)
        if cur.time.size < n_days or prev.time.size < n_days:
            continue

        seg = np.concatenate([
            np.array(prev)[n_days - N_PREV:n_days],
            np.array(cur)[0:n_days - N_PREV]
        ])
        rm = running_mean_1d(seg, RM_WIN)

        ax.plot(x, rm, color=cols[i], lw=2.5, label=f"{y:04d}")

    # baseline raw (no RM)
    ax.fill_between(x, all_mean - all_std, all_mean + all_std,
                    color="0.7", alpha=0.5)
    ax.plot(x, all_mean, "k--", lw=2)

    ax.set_xticks([0,31,61,91,122,150,181,211,242,273,304,334])
    ax.set_xticklabels(["Oct","Nov","Dec","Jan","Feb","Mar",
                        "Apr","May","Jun","Jul","Aug","Sep"])
    ax.legend()
    ax.set_ylabel(f"EHF 45–75°N ({level_label})")
    ax.set_title("Special Years vs EXTR Climatology (15d RM)")

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close()


# ==========================================================
# Main
# ==========================================================
def main_blockB():
    ehf_extr = xr.load_dataarray(EHF_EXTR_NC)
    ehf_merg = xr.load_dataarray(EHF_MERGED_NC)

    lev_extr_hpa = ehf_extr["lev"].values
    lev_merg_pa  = ehf_merg["lev"].values

    low10 = np.loadtxt(LOW10_TXT, dtype=int)
    low25 = np.loadtxt(LOW25_TXT, dtype=int)

    target_hpa = [10, 50, 100]

    for l in target_hpa:
        # pick level index
        idx_extr = np.argmin(np.abs(lev_extr_hpa - l))
        idx_merg = get_level_index_pa(lev_merg_pa, l)

        var_extr = ehf_extr.isel(lev=idx_extr)
        var_merg = ehf_merg.isel(lev=idx_merg)

        label = f"{l} hPa"

        # -------- low10 raw --------
        base = f"EHF_45_75N_{l}hPa_low10"
        plot_low10_vs_others_raw(
            var_extr, low10, label,
            os.path.join(EHF_ROOT, base + "_raw.png"),
            os.path.join(EHF_ROOT, base + "_raw.pdf"),
        )

        # -------- low10 RM --------
        plot_low10_vs_others_RM(
            var_extr, low10, label,
            os.path.join(EHF_ROOT, base + "_RM.png"),
            os.path.join(EHF_ROOT, base + "_RM.pdf"),
        )

        # -------- special years vs climatology (RM only) --------
        base2 = f"EHF_45_75N_{l}hPa_special_vs_clim"
        plot_special_vs_clim_RM(
            var_extr, var_merg, low25, label,
            os.path.join(EHF_ROOT, base2 + "_RM.png"),
            os.path.join(EHF_ROOT, base2 + "_RM.pdf"),
        )

    print("[BlockB] Done.")


if __name__ == "__main__":
    main_blockB()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
BlockC_EHF (zonal-mean, 45–75N)

目标：
  针对特殊年份 0008/0013/0014/0019，计算：
    - 45–75N 区域平均 v'T' 的 standardized anomaly (σ)
    - 基于 EXTR 200 年气候态的 t-test + bootstrap 显著性
  时间轴：
    - composite: [prev Oct–Dec(y-1), cur Jan–Sep(y)] → 365 天

输入：
  - EHF_EXTR_45_75N_lev_time.nc  (BlockA 输出：EXTR 200 年 v'T' 45–75N)
  - BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc  (merged INT, 含 V/T)

输出：
  - EHF_vert_std_anom_special_45_75N.nc
"""

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

ROOT_EHF = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(ROOT_EHF, exist_ok=True)

# baseline: BlockA_EHF 写出的 45–75N EHF
EHF_EXTR_45_75N_NC = os.path.join(ROOT_EHF, "EHF_EXTR_45_75N_lev_time.nc")

# merged INT 文件（含 V/T）
INT_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

OUT_NC = os.path.join(ROOT_EHF, "EHF_vert_std_anom_special_45_75N.nc")

YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)
N_PREV = 91          # 前一年 Oct–Dec 天数
ALPHA = 0.05
NBOOT = 3000         # bootstrap 次数（可以按需改大/改小）


def bootstrap_ci(data, nboot=NBOOT, alpha=ALPHA, rng=None):
    """1D 样本 data 的 bootstrap 均值置信区间"""
    data = np.asarray(data)
    data = data[np.isfinite(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan
    if rng is None:
        rng = np.random.default_rng()
    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)
    lo = np.percentile(means, 100 * alpha / 2.0)
    hi = np.percentile(means, 100 * (1 - alpha / 2.0))
    return lo, hi


def compute_significance_for_baseline(
    base_anom,     # DataArray(time, lev)
    anom_ref,      # ndarray (lev, t_len) 当前 ref_year 的 anomaly
    doy_base,      # baseline 每个 time 的 dayofyear (time,)
    doy_comp,      # composite days 的 dayofyear (t_len,)
    alpha=ALPHA,
    nboot=NBOOT,
):
    """
    对每个 (lev, t)：
      - 从 baseline 中取同一 DOY 的所有样本，形成列 col
      - 用 col 对 obs=anom_ref[lev,t] 做 t-test + bootstrap
    返回：
      - t_sig(lev, t), b_sig(lev, t)  (bool)
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue
        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[np.isfinite(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]
            if not np.isfinite(obs):
                continue

            # t-test
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            # bootstrap CI
            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def area_mean_45_75N(da):
    """对 V'T'(time, lev, lat) 在 45–75N 做 cos(lat) 加权平均。"""
    if "lat" not in da.dims:
        raise ValueError(f"area_mean_45_75N: no 'lat' in dims={da.dims}")
    da_band = da.sel(lat=slice(45.0, 75.0))
    w = np.cos(np.deg2rad(da_band["lat"]))
    return da_band.weighted(w).mean("lat")


def calc_eddy_heat_flux_from_INT():
    """
    从 INT_FILE 中读取 V/T，并计算 v'T' (time, plev, lat)。
    只计算 special 年及其前一年，节省时间。
    返回：
      vt_by_year: dict[year] = DataArray(time, plev, lat)
      plev: np.ndarray (Pa)
    """
    print("[BlockC_EHF] Reading INT file:", INT_FILE)
    ds_int = xr.open_dataset(INT_FILE)

    V = ds_int["V"]
    T = ds_int["T"]
    plev = ds_int["plev"].values  # Pa

    years_int = ds_int.time.dt.year.values.astype(int)
    all_years = np.unique(years_int)
    print("[BlockC_EHF] INT years available:", all_years)

    vt_by_year = {}

    for y in YEARS_SPECIAL:
        for yy in [y - 1, y]:
            if yy in vt_by_year:
                continue
            if yy not in all_years:
                print(f"[BlockC_EHF] ⚠️ Year {yy:04d} not found in INT, skip.")
                continue

            print(f"[BlockC_EHF] Computing v'T' for year {yy:04d} …")
            V_y = V.sel(time=ds_int.time.dt.year == yy)
            T_y = T.sel(time=ds_int.time.dt.year == yy)

            V_zm = V_y.mean("lon")
            T_zm = T_y.mean("lon")
            V_prime = V_y - V_zm
            T_prime = T_y - T_zm

            vt_eddy = (V_prime * T_prime).mean("lon")  # (time, plev, lat)
            vt_by_year[yy] = vt_eddy

    ds_int.close()
    return vt_by_year, plev


def main_blockC():
    # ===== 1. baseline: EXTR 45–75N =====
    print("[BlockC_EHF] Loading baseline from:", EHF_EXTR_45_75N_NC)
    ehf_extr = xr.load_dataarray(EHF_EXTR_45_75N_NC)  # (time, lev)

    lev_keep = ehf_extr["lev"].values.astype(float)
    lev_n = lev_keep.size

    time_extr = ehf_extr["time"]
    years_extr = np.unique(time_extr.dt.year.values.astype(int))
    doy_extr = time_extr.dt.dayofyear.values.astype(int)
    print("[BlockC_EHF] EXTR years:", years_extr[0], "→", years_extr[-1])

    n_days = int(time_extr.dt.dayofyear.max())
    print("[BlockC_EHF] Days per year (EXTR):", n_days)

    # climatology mean/std by DOY
    clim_all = ehf_extr.groupby("time.dayofyear").mean("time")  # (dayofyear, lev)
    clim_std = ehf_extr.groupby("time.dayofyear").std("time")

    # baseline anomaly: 每个样本相对其 DOY 气候态
    clim_all_sel_for_extr = clim_all.sel(dayofyear=time_extr.dt.dayofyear)
    base_anom_all = ehf_extr - clim_all_sel_for_extr  # (time, lev)

    # composite DOY sequence: [prev Oct–Dec | cur Jan–Sep]
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
        np.arange(1, n_days - N_PREV + 1, dtype=int),
    ])
    t_len = doy_comp.size
    print("[BlockC_EHF] Composite DOY sequence:", doy_comp[:5], "…", doy_comp[-5:])

    clim_all_comp = clim_all.sel(dayofyear=doy_comp)
    clim_std_comp = clim_std.sel(dayofyear=doy_comp)
    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values  # (lev,t)
    clim_std_comp_vals = clim_std_comp.transpose("lev", "dayofyear").values

    # ===== 2. INT special years: 45–75N v'T' =====
    vt_by_year, plev_int = calc_eddy_heat_flux_from_INT()
    plev_hpa_int = plev_int / 100.0  # Pa → hPa

    lev_keep_da = xr.DataArray(lev_keep, dims=("lev",))

    # 输出数组
    n_ref = len(YEARS_SPECIAL)
    std_anom_arr = np.full((n_ref, lev_n, t_len), np.nan, dtype=float)
    t_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        print(f"\n[BlockC_EHF] ==== Special year {y:04d} ====")

        if (y not in vt_by_year) or ((y - 1) not in vt_by_year):
            print(f"[BlockC_EHF] ⚠️ Year {y:04d} or {y-1:04d} missing v'T', skip.")
            continue

        vt_cur = vt_by_year[y]      # (time, plev, lat)
        vt_prev = vt_by_year[y - 1]

        if vt_cur.sizes["time"] < n_days or vt_prev.sizes["time"] < n_days:
            print(f"[BlockC_EHF] ⚠️ Year {y:04d} or {y-1:04d} has <{n_days} days, skip.")
            continue

        # 区域平均 45–75N
        vt_cur_45_75 = area_mean_45_75N(vt_cur)   # (time, plev)
        vt_prev_45_75 = area_mean_45_75N(vt_prev)

        # 垂直插值到 baseline lev (hPa)
        vt_cur_lev = vt_cur_45_75.assign_coords(
            plev_hpa=("plev", plev_hpa_int)
        ).swap_dims({"plev": "plev_hpa"})
        vt_prev_lev = vt_prev_45_75.assign_coords(
            plev_hpa=("plev", plev_hpa_int)
        ).swap_dims({"plev": "plev_hpa"})

        vt_cur_interp = vt_cur_lev.interp(plev_hpa=lev_keep_da)
        vt_prev_interp = vt_prev_lev.interp(plev_hpa=lev_keep_da)

        vt_cur_vals = vt_cur_interp.transpose("time", "lev").values  # (time,lev)
        vt_prev_vals = vt_prev_interp.transpose("time", "lev").values

        # 复合：prev Oct–Dec + cur Jan–Sep → (t_len, lev)，再转成 (lev,t_len)
        ref_comp_vals = np.concatenate(
            [
                vt_prev_vals[n_days - N_PREV:n_days, :],
                vt_cur_vals[0:n_days - N_PREV, :],
            ],
            axis=0,
        )  # (t_len, lev)
        ref_comp = ref_comp_vals.T  # (lev, t_len)

        # anomaly & std.anom
        anom_ref = ref_comp - clim_all_comp_vals  # (lev, t_len)

        with np.errstate(divide="ignore", invalid="ignore"):
            std_anom = anom_ref / clim_std_comp_vals
            std_anom[~np.isfinite(std_anom)] = np.nan

        std_anom_arr[yi, :, :] = std_anom

        # 显著性：基于 baseline anomaly
        print(f"[BlockC_EHF]   t/boot significance for {y:04d} …")
        t_sig, b_sig = compute_significance_for_baseline(
            base_anom_all, anom_ref, doy_extr, doy_comp, alpha=ALPHA, nboot=NBOOT
        )
        t_sig_arr[yi, :, :] = t_sig
        b_sig_arr[yi, :, :] = b_sig

    # ===== 3. 写出文件 =====
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev_keep),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "std_anom": (("ref_year", "lev", "time_index"), std_anom_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_std_comp": (("lev", "time_index"), clim_std_comp_vals),
            "t_sig": (("ref_year", "lev", "time_index"), t_sig_arr),
            "b_sig": (("ref_year", "lev", "time_index"), b_sig_arr),
        },
    )

    ds_out["std_anom"].attrs["units"] = "σ"
    ds_out["clim_all_comp"].attrs["units"] = "K m s-1"
    ds_out["clim_std_comp"].attrs["units"] = "K m s-1"
    ds_out["lev"].attrs["units"] = "hPa"
    ds_out.attrs["description"] = (
        "Eddy heat flux v'T' (45–75N) standardized anomalies (σ) "
        "for special years relative to EXTR all-years climatology.\n"
        "Baseline: EHF_EXTR_45_75N_lev_time.nc; no running mean applied."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC_EHF] Saved dataset to: {OUT_NC}")
    print("[BlockC_EHF] Done.")


if __name__ == "__main__":
    main_blockC()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (EHF) — 绘制 std.anom 垂直时间剖面
#
# 【更新要求】
# 1) raw 与 15-day RM 剖面分开成两张图绘制
# 2) 文件名增加后缀 _raw / _RM（输出路径不变）
# 3) 重要注释：用于对比的 EXTR climatology/baseline
#    在 BlockC 中没有使用 running mean（NO RM baseline）
#
# 输入：
#   EHF_vert_std_anom_special_45_75N.nc
# 输出：
#   EHF_45_75N_std_anom_refYYYY_tTest_raw.png/pdf
#   EHF_45_75N_std_anom_refYYYY_tTest_RM.png/pdf
#   EHF_45_75N_std_anom_refYYYY_boot_raw.png/pdf
#   EHF_45_75N_std_anom_refYYYY_boot_RM.png/pdf
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches

EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

IN_NC = os.path.join(EHF_ROOT, "EHF_vert_std_anom_special_45_75N.nc")
RM_WIN = 15


def running_mean_time_2d(field_lev_time, win=15, circular=False):
    """
    centered running mean along time axis for (lev, time).
    circular=False for event composites (no wrap).
    """
    assert win % 2 == 1, "running mean window must be odd for centered mean."
    field = np.asarray(field_lev_time, dtype=float)
    lev_n, t_n = field.shape
    k = win // 2
    out = np.full_like(field, np.nan)

    if circular:
        pad = np.concatenate([field[:, -k:], field, field[:, :k]], axis=1)
        for ti in range(t_n):
            w = pad[:, ti:ti+win]
            out[:, ti] = np.nanmean(w, axis=1)
    else:
        for ti in range(t_n):
            i0 = max(0, ti-k)
            i1 = min(t_n, ti+k+1)
            w = field[:, i0:i1]
            out[:, ti] = np.nanmean(w, axis=1)

    return out


def plot_one_panel(
    lev_hpa_sorted,
    field_sorted,
    sig_mask,
    ref_year,
    mode,
    tag,
    mag_threshold=1.0
):
    """
    单张图（raw 或 RM）。
    tag: "raw" 或 "RM"
    """
    n_days = field_sorted.shape[1]
    x = np.arange(n_days)

    vmin, vmax = -4.0, 4.0
    levels = np.linspace(vmin, vmax, 13)

    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.2), constrained_layout=True)

    cf = ax.contourf(
        x, lev_hpa_sorted, field_sorted,
        levels=levels, cmap="RdBu_r", extend="both"
    )

    mag_mask = np.abs(field_sorted) >= mag_threshold
    final_mask = sig_mask & mag_mask
    sig_int = final_mask.astype(int)

    ax.contourf(
        x, lev_hpa_sorted, sig_int,
        levels=[0.5, 1.5],
        colors="none", hatches=["////"],
        linewidths=0.0, alpha=0.0, zorder=5
    )

    ax.set_yscale("log")
    ax.set_ylim(850, lev_hpa_sorted.min())

    yticks_all = [1, 5, 10, 30, 50, 70, 100, 200, 300, 500, 700, 850, 1000]
    yticks = [p for p in yticks_all if (p >= lev_hpa_sorted.min() and p <= 850)]
    ax.set_yticks(yticks)
    ax.get_yaxis().set_major_formatter(mticker.ScalarFormatter())
    ax.set_ylabel("Pressure (hPa)")

    ax.set_xlim(0, n_days - 1)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "Jun", "Jul", "Aug", "Sep"]
    )
    ax.set_xlabel("Time (composite days)")

    # baseline NO RM 说明写在标题里，避免误解
    ax.set_title(
        f"{tag} std.anom — year {int(ref_year):04d} ({mode})\n"
        f"Baseline climatology: NO running mean",
        fontsize=9
    )

    cbar = plt.colorbar(cf, ax=ax, orientation="vertical", shrink=0.95)
    cbar.set_label("v'T' standardized anomaly (σ)")

    label_sig = f"significant by {mode} (p<0.05 & |σ|≥{mag_threshold})"
    hatch_patch = mpatches.Patch(facecolor="none", edgecolor="k",
                                 hatch="////", label=label_sig)
    ax.legend(handles=[hatch_patch], loc="upper right", fontsize=8, frameon=False)

    out_png = os.path.join(EHF_ROOT, f"EHF_45_75N_std_anom_ref{int(ref_year):04d}_{mode}_{tag}.png")
    out_pdf = os.path.join(EHF_ROOT, f"EHF_45_75N_std_anom_ref{int(ref_year):04d}_{mode}_{tag}.pdf")
    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print(f"[BlockD_EHF] Saved {mode} {tag} figure for year {int(ref_year):04d} to:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockD_EHF():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev_hpa = ds["lev"].values
    std_anom = ds["std_anom"]
    t_sig = ds["t_sig"]
    b_sig = ds["b_sig"]

    sort_idx = np.argsort(lev_hpa)
    lev_hpa_sorted = lev_hpa[sort_idx]

    for i, y in enumerate(ref_years):
        field = std_anom.isel(ref_year=i).values
        if np.all(~np.isfinite(field)):
            print(f"[BlockD_EHF] Year {int(y):04d} is all NaN, skip.")
            continue

        field_sorted_raw = field[sort_idx, :]
        field_sorted_rm  = running_mean_time_2d(field_sorted_raw, win=RM_WIN, circular=False)

        t_mask = t_sig.isel(ref_year=i).values[sort_idx, :]
        b_mask = b_sig.isel(ref_year=i).values[sort_idx, :]

        # t-test raw + RM（分别出图）
        plot_one_panel(lev_hpa_sorted, field_sorted_raw, t_mask, y, "tTest", "raw", mag_threshold=1.0)
        plot_one_panel(lev_hpa_sorted, field_sorted_rm,  t_mask, y, "tTest", "RM",  mag_threshold=1.0)

        # bootstrap raw + RM（分别出图）
        plot_one_panel(lev_hpa_sorted, field_sorted_raw, b_mask, y, "boot", "raw", mag_threshold=1.0)
        plot_one_panel(lev_hpa_sorted, field_sorted_rm,  b_mask, y, "boot", "RM",  mag_threshold=1.0)

    ds.close()
    print("[BlockD_EHF] Done.")


if __name__ == "__main__":
    main_blockD_EHF()


In [ ]:
#block EF pre, find the timeslot.
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
# 为平均区间做定义，属于EF的前置代码。
import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import pandas as pd

# ============================================================
# 1. 配置参数
# ============================================================
# 输出根目录
OUT_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF/Spatial_Analysis_Segmentation"
os.makedirs(OUT_ROOT, exist_ok=True)

# 输入数据路径 (假设这些路径对所有年份通用，或数据包含所有年份)
DATA_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_U"
U_EXTR_60N_NC   = os.path.join(DATA_ROOT, "U_EXTR_60N_lev_time.nc")
U_MERGED_60N_NC = os.path.join(DATA_ROOT, "U_MERGED_60N_lev_time.nc")

# 目标年份列表 (0008, 0014, 0013, 0019)
TARGET_YEARS = [8, 13, 14, 19]

N_PREV      = 91     # 前一年 Oct–Dec 天数
TARGET_HPA  = 100.0
RM_WIN      = 15     # 15天平滑窗口

# --- 核心控制参数 ---
# 只有当波动幅度 > 2.5 m/s 时，才保留该波段。
MIN_AMP = 2.5 

# --- 颜色配置 ---
COLOR_UP   = "#a6cee3"  # 浅蓝 (增强)
COLOR_DOWN = "#fb9a99"  # 浅红 (减弱)
COLOR_ANOM = "purple"   # 距平曲线颜色

# ============================================================
# 2. 核心算法：基于振幅的极值过滤
# ============================================================

def running_mean_1d(arr, win=15):
    """中心 1D running mean"""
    assert win % 2 == 1
    arr = np.asarray(arr, dtype=float)
    n = arr.size
    k = win // 2
    out = np.full(n, np.nan)
    for i in range(n):
        i0 = max(0, i - k)
        i1 = min(n, i + k + 1)
        out[i] = np.nanmean(arr[i0:i1])
    return out

def get_level_index_pa(lev_vals_pa, target_hpa):
    """找到最近的气压层索引"""
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)

def find_clean_turning_points(series_rm, min_amp):
    """
    寻找关键拐点，成对消除振幅小于 min_amp 的微小波动。
    """
    s = np.asarray(series_rm, dtype=float)
    n = s.size
    
    # 1. 初始所有极值
    raw_tps = []
    raw_tps.append({"idx": 0, "val": s[0], "type": "start"})
    
    for i in range(1, n - 1):
        if np.isnan(s[i-1]) or np.isnan(s[i]) or np.isnan(s[i+1]): continue
        if (s[i] >= s[i-1]) and (s[i] > s[i+1]):
            raw_tps.append({"idx": i, "val": s[i], "type": "max"})
        elif (s[i] <= s[i-1]) and (s[i] < s[i+1]):
            raw_tps.append({"idx": i, "val": s[i], "type": "min"})
            
    raw_tps.append({"idx": n-1, "val": s[n-1], "type": "end"})
    
    # 2. 循环过滤
    while True:
        changed = False
        if len(raw_tps) <= 2: break
        
        new_tps = []
        new_tps.append(raw_tps[0]) # 保留起点
        
        i = 1
        while i < len(raw_tps) - 1:
            curr = raw_tps[i]
            nxt  = raw_tps[i+1]
            diff = abs(curr["val"] - nxt["val"])
            
            # 如果波幅太小，同时删除 curr 和 nxt (成对消除)
            if diff < min_amp:
                changed = True
                i += 2 
            else:
                new_tps.append(curr)
                i += 1
                
        # 处理尾部
        if i == len(raw_tps) - 1:
            new_tps.append(raw_tps[i])
        
        # 确保终点在
        if new_tps[-1]["idx"] != (n-1):
             new_tps.append(raw_tps[-1])
             
        raw_tps = new_tps
        if not changed:
            break
            
    return raw_tps

def build_stages(tps, time_axis=None):
    """
    构建阶段列表。
    如果提供了 time_axis (xarray Time 或 numpy array)，会转换成日期字符串。
    """
    stages = []
    for i in range(len(tps) - 1):
        idx_s = tps[i]["idx"]
        idx_e = tps[i+1]["idx"]
        val_s = tps[i]["val"]
        val_e = tps[i+1]["val"]
        
        diff = val_e - val_s
        state = 1 if diff > 0 else -1
        
        # 获取日期字符串 (如果提供了时间轴)
        date_s_str = ""
        date_e_str = ""
        if time_axis is not None:
            # 假设时间轴是 xarray 的 datetime64[ns]
            try:
                date_s_str = str(time_axis[idx_s].values)[:10]
                date_e_str = str(time_axis[idx_e].values)[:10]
            except:
                pass

        stages.append({
            "idx_start": idx_s,
            "idx_end": idx_e,
            "length": idx_e - idx_s,
            "state": state,
            "diff": diff,
            "date_start": date_s_str,
            "date_end": date_e_str
        })
    return stages

# ============================================================
# 3. 主程序
# ============================================================

def main():
    print(f"--- Processing Years: {TARGET_YEARS} ---")
    print(f"--- Output Dir: {OUT_ROOT} ---")
    
    # 1. 载入一次数据 (避免重复IO)
    print("Loading netCDF data...")
    u_extr = xr.load_dataarray(U_EXTR_60N_NC)
    u_merged = xr.load_dataarray(U_MERGED_60N_NC)
    
    idx_lev, lev_actual = get_level_index_pa(u_extr.lev.values, TARGET_HPA)
    print(f"Selected Level: {lev_actual:.1f} hPa")
    
    u_extr = u_extr.isel(lev=idx_lev)
    u_merged = u_merged.isel(lev=idx_lev)
    
    # 2. 计算气候态 (Climatology) - 只需要算一次
    n_days = 365 
    clim_mean = u_extr.groupby("time.dayofyear").mean("time").values[:n_days]
    clim_std  = u_extr.groupby("time.dayofyear").std("time").values[:n_days]
    
    comp_clim_mean = np.concatenate([clim_mean[n_days-N_PREV:], clim_mean[:n_days-N_PREV]])
    comp_clim_std  = np.concatenate([clim_std[n_days-N_PREV:], clim_std[:n_days-N_PREV]])
    comp_clim_rm   = running_mean_1d(comp_clim_mean, win=RM_WIN)
    
    # 用于收集所有年份的 summary
    all_years_summary = []

    # 3. 循环处理年份
    for target_year in TARGET_YEARS:
        print(f"\n=== Processing Year {target_year:04d} ===")
        
        # --- 3.1 提取数据 ---
        try:
            u_prev = u_merged.sel(time=u_merged.time.dt.year == (target_year - 1))
            u_cur  = u_merged.sel(time=u_merged.time.dt.year == target_year)
            
            # 拼接数据值
            u_prev_vals = u_prev.values
            u_cur_vals  = u_cur.values
            comp_u_raw = np.concatenate([u_prev_vals[365-N_PREV:365], u_cur_vals[0:365-N_PREV]])
            
            # 拼接时间轴 (用于记录日期)
            time_prev = u_prev.time
            time_cur  = u_cur.time
            # 注意 xarray concat
            comp_time = xr.concat([
                time_prev.isel(time=slice(365-N_PREV, 365)),
                time_cur.isel(time=slice(0, 365-N_PREV))
            ], dim="time")

        except Exception as e:
            print(f"Skipping Year {target_year}: Data missing or insufficient. Error: {e}")
            continue

        n_comp = comp_u_raw.size
        x_axis = np.arange(n_comp)
        
        # --- 3.2 计算 ---
        comp_u_rm = running_mean_1d(comp_u_raw, win=RM_WIN)
        u_anomaly = comp_u_rm - comp_clim_rm
        
        clean_tps = find_clean_turning_points(comp_u_rm, min_amp=MIN_AMP)
        stages = build_stages(clean_tps, time_axis=comp_time)
        
        print(f"Found {len(stages)} stages.")
        
        # --- 3.3 记录到 Summary ---
        for i, st in enumerate(stages):
            # 将每一段的信息存入列表
            stage_type_str = "UP" if st['state'] == 1 else "DOWN"
            all_years_summary.append({
                "Year": target_year,
                "Stage_ID": i + 1,
                "Type": stage_type_str,
                "Start_Idx": st['idx_start'],
                "End_Idx": st['idx_end'],
                "Start_Date": st['date_start'],
                "End_Date": st['date_end'],
                "Length_Days": st['length'],
                "Diff_U": round(st['diff'], 2)
            })

        # --- 3.4 绘图 ---
        fig, ax1 = plt.subplots(1, 1, figsize=(14, 6), constrained_layout=True)
        
        # [左轴] 
        line_clim_shade = ax1.fill_between(x_axis, comp_clim_mean-comp_clim_std, comp_clim_mean+comp_clim_std,
                                         color="0.92", label="Climatology ±1σ", zorder=0)
        line_clim_mean, = ax1.plot(x_axis, comp_clim_mean, color="gray", ls="--", lw=1.5, alpha=0.6, label="Clim Mean")
        line_raw,       = ax1.plot(x_axis, comp_u_raw, color="tab:blue", ls="--", lw=1.2, alpha=0.4, label="Raw Wind")
        line_rm,        = ax1.plot(x_axis, comp_u_rm, color="tab:blue", lw=2.5, label="15d RM Wind", zorder=5)
        
        for st in stages:
            color = COLOR_UP if st["state"] == 1 else COLOR_DOWN
            ax1.axvspan(st["idx_start"], st["idx_end"], facecolor=color, alpha=0.35, zorder=-1)
            
        # [右轴]
        ax2 = ax1.twinx()
        line_anom, = ax2.plot(x_axis, u_anomaly, color=COLOR_ANOM, ls=":", lw=2, alpha=0.9, label="Wind Anomaly")
        ax2.axhline(0, color=COLOR_ANOM, lw=0.5, alpha=0.3)
        
        # [装饰]
        ax1.set_xlim(0, n_comp-1)
        ax1.set_ylabel(f"Zonal Mean U @ {lev_actual:.0f}hPa (m/s)", fontsize=12, color="tab:blue")
        ax2.set_ylabel("Anomaly (m/s)", fontsize=12, color=COLOR_ANOM)
        ax2.spines["right"].set_color(COLOR_ANOM)
        ax2.tick_params(axis='y', colors=COLOR_ANOM)
        
        xticks = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
        xticklabels = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep"]
        ax1.set_xticks(xticks)
        ax1.set_xticklabels(xticklabels, fontsize=11)
        
        ax1.set_title(f"Year {target_year:04d} Wind Regimes + Anomaly\n"
                      f"Filter Amp > {MIN_AMP} m/s", fontsize=14, fontweight="bold", pad=12)
        
        # [图例]
        patch_up = Patch(facecolor=COLOR_UP, edgecolor='none', alpha=0.35, label="Strengthening")
        patch_dn = Patch(facecolor=COLOR_DOWN, edgecolor='none', alpha=0.35, label="Weakening")
        final_h = [line_rm, line_raw, line_clim_mean, line_anom, patch_up, patch_dn]
        final_l = ["Wind (15d RM)", "Raw Wind", "Climatology", "Anomaly", "Strengthening", "Weakening"]
        
        ax1.legend(final_h, final_l, loc="upper right", ncol=3, 
                   frameon=True, framealpha=1.0, facecolor="white", edgecolor="0.8", fontsize=10)

        # [保存图片]
        out_png = os.path.join(OUT_ROOT, f"U_60N_{int(lev_actual)}hPa_Year{target_year:04d}_Segmentation.png")
        plt.savefig(out_png, dpi=300)
        plt.close(fig) # 关闭图表释放内存
        print(f"Saved plot: {out_png}")

    # --- 4. 保存 TXT 汇总文件 ---
    if all_years_summary:
        txt_path = os.path.join(OUT_ROOT, "Wind_Stages_Summary.txt")
        print(f"\nWriting summary to: {txt_path}")
        
        # 使用 pandas 格式化输出，对齐更漂亮
        df = pd.DataFrame(all_years_summary)
        # 调整列顺序
        cols = ["Year", "Stage_ID", "Type", "Start_Date", "End_Date", "Length_Days", "Diff_U", "Start_Idx", "End_Idx"]
        df = df[cols]
        
        # 保存为 CSV 格式 (虽然扩展名是 txt，逗号分隔方便程序读)
        # 或者使用 to_string 保存为类似表格的文本
        with open(txt_path, 'w') as f:
            f.write(f"# Wind Segmentation Summary\n")
            f.write(f"# Level: {lev_actual:.1f} hPa\n")
            f.write(f"# RM_Window: {RM_WIN} days, Min_Amp: {MIN_AMP} m/s\n")
            f.write(f"# Created by segmentation script\n")
            f.write("-" * 80 + "\n")
            # 写入对齐的表格
            f.write(df.to_string(index=False))
        
        print("Done.")

if __name__ == "__main__":
    main()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
============================================================
BlockE — NH 100 hPa EHF stage composites
------------------------------------------------------------
功能：
  - 读取 EF pre 输出的 Wind_Stages_Summary.txt
  - 读取 BlockA 输出的 EHF_MERGED_vTprime_4D.nc（v'T' 4D）
  - 在 100 hPa、北半球（lat >= 0）上，按 EF pre 的分段
    （Start_Idx, End_Idx）做时间平均，得到空间 composite

输入：
  1) Wind_Stages_Summary.txt
     - 来自 EF pre 脚本
     - 包含 Year, Stage_ID, Type, Start_Idx, End_Idx 等列
  2) EHF_MERGED_vTprime_4D.nc
     - 来自 BlockA
     - dims: (time, plev, lat, lon)
     - 垂直坐标名为 plev（单位 Pa）

输出：
  - EHF_100hPa_NH_stage_composites.nc

    结构：
      dims: entry, lat, lon
      data: EHF_mean(entry, lat, lon)

      coords (along entry):
        year        (entry) int
        stage_id    (entry) int
        type        (entry) string, "UP"/"DOWN"
        start_idx   (entry) int   # composite 索引区间 [start_idx, end_idx)
        end_idx     (entry) int
        length_days (entry) int
        diff_u      (entry) float
        start_date  (entry) string  # 文本中记录的日期
        end_date    (entry) string

    说明：
      - 时间 composite 构造方式与 EF pre 一致：
        [prev year Oct–Dec 的最后 N_PREV 天] +
        [current year Jan–Sep 的前 (365-N_PREV) 天]
      - 对应的 EHF composite 在该 time 轴上做平均。
============================================================
"""

import os
import numpy as np
import xarray as xr
import pandas as pd

# ---------------- 路径配置 ----------------
ROOT_EHF = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
MERGED_FULL_NC = os.path.join(ROOT_EHF, "EHF_MERGED_vTprime_4D.nc")

SEG_ROOT = os.path.join(ROOT_EHF, "Spatial_Analysis_Segmentation")
os.makedirs(SEG_ROOT, exist_ok=True)

STAGE_SUMMARY_TXT = os.path.join(SEG_ROOT, "Wind_Stages_Summary.txt")

OUT_NC = os.path.join(SEG_ROOT, "EHF_100hPa_NH_stage_composites.nc")

# ---------------- 常数配置 ----------------
TARGET_YEARS = [8, 13, 14, 19]   # 要处理的特殊年份
N_PREV = 91                      # 与 EF pre 保持一致
TARGET_HPA = 100.0               # 只做 100 hPa
HEMI_LAT_MIN = 0.0               # 北半球 lat >= 0


# =========================================================
# 读取 Wind_Stages_Summary.txt
# =========================================================
def read_stage_summary(path_txt: str) -> pd.DataFrame:
    """解析 EF pre 生成的 Wind_Stages_Summary.txt，返回 DataFrame。"""
    if not os.path.exists(path_txt):
        raise FileNotFoundError(f"[BlockE] Stage summary file not found: {path_txt}")

    print(f"[BlockE] Reading stage summary from: {path_txt}")
    with open(path_txt, "r") as f:
        raw_lines = f.readlines()

    # 去掉注释和空行
    cleaned = []
    for line in raw_lines:
        if line.strip() == "":
            continue
        if line.lstrip().startswith("#"):
            continue
        cleaned.append(line.rstrip("\n"))

    if not cleaned:
        raise ValueError("[BlockE] After removing comments/blank lines, file is empty.")

    # 找 header 行（含 Year 和 Stage_ID）
    header_idx = None
    for i, line in enumerate(cleaned):
        if "Year" in line and "Stage_ID" in line:
            header_idx = i
            break
    if header_idx is None:
        raise ValueError("[BlockE] Cannot find header line with 'Year' and 'Stage_ID'.")

    from io import StringIO
    table_str = "\n".join(cleaned[header_idx:])
    buf = StringIO(table_str)
    df = pd.read_fwf(buf)

    # 列名去空格
    df.columns = [str(c).strip() for c in df.columns]

    expected_cols = [
        "Year", "Stage_ID", "Type", "Start_Date", "End_Date",
        "Length_Days", "Diff_U", "Start_Idx", "End_Idx"
    ]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(
            f"[BlockE] Missing columns in stage summary: {missing}\n"
            f"Parsed columns: {list(df.columns)}"
        )

    print("[BlockE] Stage summary head:")
    print(df.head())

    return df


# =========================================================
# 垂直坐标/层选择
# =========================================================
def find_vertical_coord(vt_full: xr.DataArray) -> str:
    """自动识别垂直坐标名（lev / plev / 其它）。"""
    if "lev" in vt_full.coords:
        lev_coord_name = "lev"
    elif "plev" in vt_full.coords:
        lev_coord_name = "plev"
    else:
        lev_candidates = [d for d in vt_full.dims if d not in ("time", "lat", "lon")]
        if len(lev_candidates) == 1:
            lev_coord_name = lev_candidates[0]
            print(f"[BlockE] ⚠️ No 'lev' or 'plev'; using dim '{lev_coord_name}' as vertical coord.")
        else:
            raise ValueError(
                "[BlockE] Cannot determine vertical coordinate name. "
                f"dims={vt_full.dims}, coords={list(vt_full.coords)}"
            )
    print(f"[BlockE] Using vertical coordinate: {lev_coord_name}")
    return lev_coord_name


def find_level_index_hpa(lev_vals, target_hpa):
    """lev_vals 可以是 Pa 或 hPa，返回最接近 target_hpa 的索引和实际 hPa 值。"""
    lev_vals = np.asarray(lev_vals, dtype=float)
    if np.nanmax(lev_vals) > 2000.0:
        lev_hpa = lev_vals / 100.0
    else:
        lev_hpa = lev_vals
    idx = int(np.argmin(np.abs(lev_hpa - target_hpa)))
    return idx, lev_hpa[idx]


# =========================================================
# 构造与 EF pre 一致的 composite（prevOct–Dec + curJan–Sep）
# =========================================================
def build_vt_composite_for_year(vt_lev_NH: xr.DataArray,
                                year: int,
                                n_prev: int = N_PREV) -> xr.DataArray:
    """
    vt_lev_NH : DataArray(time, lat, lon)
    year      : 当前年，如 8, 13, 14, 19
    n_prev    : 前一年取多少天（默认 91）

    返回：
      comp : DataArray(time, lat, lon) 长度 365
      时间顺序：prev-year 最后 n_prev 天 + 当前年前 (365-n_prev) 天
    """
    vt_prev = vt_lev_NH.sel(time=vt_lev_NH.time.dt.year == (year - 1))
    vt_cur  = vt_lev_NH.sel(time=vt_lev_NH.time.dt.year == year)

    if vt_prev.sizes["time"] == 0 or vt_cur.sizes["time"] == 0:
        print(f"[BlockE] ⚠️ Year {year:04d}: prev or current year has no data.")
        return None

    n_prev_days = vt_prev.sizes["time"]
    n_cur_days  = vt_cur.sizes["time"]

    if n_prev_days != n_cur_days:
        print(f"[BlockE] ⚠️ Year {year:04d}: prev={n_prev_days}, cur={n_cur_days}, "
              "using prev days as reference.")
    if n_prev_days < n_prev:
        print(f"[BlockE] ⚠️ Year {year:04d}: prev days < N_PREV={n_prev}, skip.")
        return None

    print(f"[BlockE] Year {year:04d}: n_days={n_prev_days}, N_PREV={n_prev}")

    comp_prev = vt_prev.isel(time=slice(n_prev_days - n_prev, n_prev_days))
    comp_cur  = vt_cur.isel(time=slice(0, n_prev_days - n_prev))

    # 关键：沿 "time" 维拼接，得到长度 365 的一维时间轴
    comp = xr.concat([comp_prev, comp_cur], dim="time")

    print(f"[BlockE] Year {year:04d}: composite shape = {comp.shape}")
    return comp  # (time=365, lat, lon)


# =========================================================
# 主程序
# =========================================================
def main_blockE():
    # 1. 读取 wind segmentation summary
    df_stage = read_stage_summary(STAGE_SUMMARY_TXT)

    # 只保留目标年份
    df_stage = df_stage[df_stage["Year"].isin(TARGET_YEARS)].copy()
    df_stage.sort_values(["Year", "Stage_ID"], inplace=True)
    df_stage.reset_index(drop=True, inplace=True)

    if df_stage.empty:
        print("[BlockE] ⚠️ No stages for target years, nothing to do.")
        return

    print(f"[BlockE] Using {len(df_stage)} stages across years "
          f"{sorted(df_stage['Year'].unique())}")

    # 2. 打开 vt_full（v'T' 4D）
    if not os.path.exists(MERGED_FULL_NC):
        raise FileNotFoundError(f"[BlockE] vt_full file not found: {MERGED_FULL_NC}")

    print(f"[BlockE] Opening vt_full from: {MERGED_FULL_NC}")
    ds_vt = xr.open_dataset(MERGED_FULL_NC)
    data_vars = list(ds_vt.data_vars)
    if len(data_vars) == 0:
        raise ValueError("[BlockE] No data variables in vt_full dataset.")

    vt_name = data_vars[0]
    vt_full = ds_vt[vt_name]  # (time, plev, lat, lon)
    print(f"[BlockE] Using variable: {vt_name}, dims = {vt_full.dims}, shape = {vt_full.shape}")

    # 垂直层：100 hPa
    lev_coord = find_vertical_coord(vt_full)
    lev_vals = vt_full[lev_coord].values
    idx_lev, lev_used = find_level_index_hpa(lev_vals, TARGET_HPA)
    print(f"[BlockE] 100 hPa level index = {idx_lev}, lev_used ≈ {lev_used:.2f} hPa")

    vt_lev = vt_full.isel({lev_coord: idx_lev})  # (time, lat, lon)

    # 北半球
    if "lat" not in vt_lev.dims:
        raise ValueError("[BlockE] vt_lev has no 'lat' dimension.")
    vt_lev_NH = vt_lev.sel(lat=slice(HEMI_LAT_MIN, 90.0))
    print(f"[BlockE] vt_lev_NH shape (time, lat, lon): {vt_lev_NH.shape}")

    # 检查每年天数（只做一次 sanity check）
    years_all = np.unique(vt_lev_NH.time.dt.year.values.astype(int))
    ref_year = years_all[0]
    n_days_ref = vt_lev_NH.sel(time=vt_lev_NH.time.dt.year == ref_year).sizes["time"]
    print(f"[BlockE] Reference year {ref_year:04d} has {n_days_ref} days.")

    # 3. 为每个 target year 构建 composite
    composites_by_year = {}
    for y in sorted(df_stage["Year"].unique()):
        comp = build_vt_composite_for_year(vt_lev_NH, y, n_prev=N_PREV)
        if comp is None:
            print(f"[BlockE] ⚠️ Year {y:04d}: composite is None, skip all its stages.")
            continue
        composites_by_year[y] = comp

    if not composites_by_year:
        print("[BlockE] ⚠️ No composites constructed, abort.")
        ds_vt.close()
        return

    # 4. 按 stage 切片并时间平均
    entries = []
    fields = []

    for idx, row in df_stage.iterrows():
        year = int(row["Year"])
        if year not in composites_by_year:
            print(f"[BlockE] ⚠️ Year {year:04d} has no composite, skip its Stage {row['Stage_ID']}.")
            continue

        comp = composites_by_year[year]  # (time, lat, lon)
        t_len = comp.sizes["time"]

        start_idx = int(row["Start_Idx"])
        end_idx   = int(row["End_Idx"])

        # 范围检查
        if not (0 <= start_idx < t_len):
            print(f"[BlockE] ⚠️ Stage row {idx}: start_idx {start_idx} out of [0,{t_len}), skip.")
            continue
        if not (0 < end_idx <= t_len):
            print(f"[BlockE] ⚠️ Stage row {idx}: end_idx {end_idx} out of (0,{t_len}], skip.")
            continue
        if start_idx >= end_idx:
            print(f"[BlockE] ⚠️ Stage row {idx}: start_idx >= end_idx, skip.")
            continue

        stage_id = int(row["Stage_ID"])
        stype    = str(row["Type"])
        len_days = int(row["Length_Days"])
        diff_u   = float(row["Diff_U"])
        s_date   = str(row["Start_Date"])
        e_date   = str(row["End_Date"])

        print(f"[BlockE] Year {year:04d}, Stage {stage_id}: "
              f"idx[{start_idx},{end_idx}) len={len_days}, Type={stype}, Diff_U={diff_u:.2f}")

        seg = comp.isel(time=slice(start_idx, end_idx))  # (time_seg, lat, lon)
        field_mean = seg.mean("time", keep_attrs=False)  # (lat, lon)

        entries.append({
            "year": year,
            "stage_id": stage_id,
            "type": stype,
            "start_idx": start_idx,
            "end_idx": end_idx,
            "length_days": len_days,
            "diff_u": diff_u,
            "start_date": s_date,
            "end_date": e_date,
        })
        fields.append(field_mean)

    ds_vt.close()

    if not fields:
        print("[BlockE] ⚠️ No stage composites produced, nothing to save.")
        return

    # 5. 堆叠输出
    print(f"[BlockE] Stacking {len(fields)} stage composites.")
    lat = fields[0]["lat"]
    lon = fields[0]["lon"]
    data = np.stack([f.values for f in fields], axis=0)  # (entry, lat, lon)

    N = len(entries)
    years       = np.array([e["year"]        for e in entries], dtype=int)
    stage_ids   = np.array([e["stage_id"]    for e in entries], dtype=int)
    types       = np.array([e["type"]        for e in entries], dtype="U8")
    start_idx_a = np.array([e["start_idx"]   for e in entries], dtype=int)
    end_idx_a   = np.array([e["end_idx"]     for e in entries], dtype=int)
    length_days = np.array([e["length_days"] for e in entries], dtype=int)
    diff_u_a    = np.array([e["diff_u"]      for e in entries], dtype=float)
    start_date  = np.array([e["start_date"]  for e in entries], dtype="U16")
    end_date    = np.array([e["end_date"]    for e in entries], dtype="U16")

    da_ehf = xr.DataArray(
        data,
        dims=("entry", "lat", "lon"),
        coords={
            "entry": np.arange(N),
            "lat": lat,
            "lon": lon,
        },
        name="EHF_mean",
    )

    ds_out = xr.Dataset(
        data_vars={
            "EHF_mean": da_ehf
        },
        coords={
            "entry": np.arange(N),
            "lat": lat,
            "lon": lon,
            "year": ("entry", years),
            "stage_id": ("entry", stage_ids),
            "type": ("entry", types),
            "start_idx": ("entry", start_idx_a),
            "end_idx": ("entry", end_idx_a),
            "length_days": ("entry", length_days),
            "diff_u": ("entry", diff_u_a),
            "start_date": ("entry", start_date),
            "end_date": ("entry", end_date),
        },
        attrs={
            "description": (
                "NH (lat>=0) 100 hPa EHF (v'T') stage composites.\n"
                "Time-mean over EF-pre-defined stages [Start_Idx, End_Idx) "
                "on composite Oct–Sep timeline (prev-year Oct–Dec + cur-year Jan–Sep)."
            ),
            "level_hpa": float(lev_used),
            "N_PREV": N_PREV,
        },
    )

    print(f"[BlockE] Saving composites to: {OUT_NC}")
    ds_out.to_netcdf(OUT_NC)
    print("[BlockE] Done.")


if __name__ == "__main__":
    main_blockE()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import cartopy.crs as ccrs
import cartopy.feature as cfeature


ROOT_EHF = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
SEG_ROOT = os.path.join(ROOT_EHF, "Spatial_Analysis_Segmentation")
IN_NC = os.path.join(SEG_ROOT, "EHF_100hPa_NH_stage_composites.nc")
OUT_DIR = os.path.join(SEG_ROOT, "plots")
os.makedirs(OUT_DIR, exist_ok=True)

TARGET_YEARS = [8, 13, 14, 19]

# 固定色标范围
VMIN, VMAX = -100.0, 100.0


def make_polar_map_for_year(ds: xr.Dataset, year: int):

    mask = ds["year"].values == year
    if not np.any(mask):
        print(f"[BlockF] ⚠️ No data for year {year:04d}")
        return

    sub = ds.isel(entry=np.where(mask)[0])
    order = np.argsort(sub["stage_id"].values)
    sub = sub.isel(entry=order)

    ef   = sub["EHF_mean"]  # (entry, lat, lon)
    s_id = sub["stage_id"].values
    ttyp = sub["type"].values
    sdat = sub["start_date"].values
    edat = sub["end_date"].values
    lday = sub["length_days"].values
    du   = sub["diff_u"].values

    n_stage = ef.sizes["entry"]
    lats = ef["lat"].values
    lons = ef["lon"].values
    lon2d, lat2d = np.meshgrid(lons, lats)

    # ============ 子图布局 ============

    if n_stage <= 3:
        nrows, ncols = 1, n_stage
    elif n_stage <= 6:
        nrows, ncols = 2, int(np.ceil(n_stage / 2))
    else:
        nrows, ncols = 3, int(np.ceil(n_stage / 3))

    fig = plt.figure(figsize=(4.2 * ncols, 4.6 * nrows))
    fig.suptitle(
        f"NH 100 hPa EHF — Year {year:04d}",
        fontsize=15, fontweight="bold", y=0.97
    )

    proj = ccrs.Orthographic(central_longitude=0.0, central_latitude=90.0)
    data_crs = ccrs.PlateCarree()

    # 统一色标对象
    im = None

    # ============ 子图循环 ============

    for i in range(n_stage):
        ax = plt.subplot(nrows, ncols, i+1, projection=proj)
        ax.set_global()

        ax.coastlines(linewidth=0.6)
        ax.add_feature(cfeature.BORDERS, linewidth=0.3)

        gl = ax.gridlines(draw_labels=False,
                          linewidth=0.3,
                          linestyle="--",
                          alpha=0.4)

        field = ef.isel(entry=i).values

        im = ax.pcolormesh(
            lon2d, lat2d, field,
            transform=data_crs,
            cmap="RdBu_r",
            vmin=VMIN, vmax=VMAX,
            shading="auto"
        )

        ax.set_title(
            f"Stage {s_id[i]} ({ttyp[i]})\n"
            f"{sdat[i]} → {edat[i]} ({lday[i]} d, ΔU={du[i]:.1f})",
            fontsize=9
        )

    # ============ 底部色标 ============

    # 预留底部区域（避免色标贴到子图上）
    fig.subplots_adjust(bottom=0.15)

    # 专用 colorbar 区域： [left, bottom, width, height]
    # 位置是图区域的一次性坐标，绝对稳定
    cbar_ax = fig.add_axes([0.20, 0.06, 0.60, 0.028])

    cbar = fig.colorbar(im, cax=cbar_ax,
                        orientation="horizontal")
    cbar.set_label("EHF (v'T') at 100 hPa  [K·m/s]")

    # 保存
    out_png = os.path.join(OUT_DIR,
                           f"EHF_100hPa_NH_Year{year:04d}_stages_circ.png")
    out_pdf = os.path.join(OUT_DIR,
                           f"EHF_100hPa_NH_Year{year:04d}_stages_circ.pdf")

    fig.savefig(out_png, dpi=300)
    fig.savefig(out_pdf)
    plt.close(fig)

    print(f"[BlockF] ✔️ Saved circular maps (fixed colorbar) for {year:04d}")


def main_blockF():
    print(f"[BlockF] Loading: {IN_NC}")
    ds = xr.open_dataset(IN_NC)

    for y in TARGET_YEARS:
        make_polar_map_for_year(ds, y)

    ds.close()
    print("[BlockF] Done.")


if __name__ == "__main__":
    main_blockF()
